# Q1(a) — brainstorming scratchpad

**This is a working file, not submission material.** Nothing here is written to be
pasted into the report; it is here so you can poke at Bayes factors numerically and
arrive at your own explanation.

Everything below uses **invented numbers**. It deliberately does *not* touch `xy.txt`
or the marginal likelihood formula — that is part (c), and the point here is to
understand what a Bayes factor *means*, independently of how you compute one.

---

## What the question actually asks

> a) For any two models $M_i$ and $M_j$, write down the Bayes factor $B_{ij}(y)$ in terms
> of the model marginal likelihoods $p(y \mid M_i), p(y \mid M_j)$ **and explain how these
> quantities may be used to compare the three models** $M_1, M_2, M_3$. If equal prior
> probabilities are assigned to the three models, **explain how to obtain posterior model
> probabilities**. — 3 Marks

Three deliverables, roughly a mark each:

| # | Deliverable | Status in your draft |
|---|---|---|
| 1 | Write down $B_{ij}$ in terms of marginal likelihoods | done — eq. (2) |
| 2 | Explain how these compare $M_1, M_2, M_3$ | **missing** |
| 3 | Posterior model probabilities under equal priors | done, arguably over-long |

So this notebook is aimed squarely at **#2**.

## Questions to answer for yourself

Work through the cells below, then try to answer these in your own words:

1. Someone hands you $B_{12}(y) = 7$ and tells you nothing else. What do you now know?
2. Why is a Bayes factor said to isolate *the evidence from the data*? What has been
   cancelled out, and why does that matter?
3. There are three models. How many distinct pairwise Bayes factors are there? How many
   of them can you choose freely? What does that imply about how many you need to report?
4. Is $B_{12} = 7$ "a lot"? What would you have to appeal to in order to answer that,
   and why is that appeal not fully objective?
5. How do the Bayes factors and the posterior model probabilities relate — is one strictly
   more informative than the other?

---
## 1. What a Bayes factor does to your beliefs

The defining relation rearranges to

$$\underbrace{\frac{\mathbb{P}(M_i \mid y)}{\mathbb{P}(M_j \mid y)}}_{\text{posterior odds}}
= \underbrace{\frac{\mathbb{P}(M_i)}{\mathbb{P}(M_j)}}_{\text{prior odds}} \times B_{ij}(y).$$

So $B_{ij}$ is a *multiplier* on odds. Run this and watch what it does.

In [1]:
def posterior_odds(prior_odds, B):
    """Bayes factors act multiplicatively on the odds scale."""
    return prior_odds * B


def odds_to_prob(odds):
    return odds / (1 + odds)


print(f"{'B_ij':>8} {'prior odds':>12} {'post. odds':>12} {'P(M_i | y)':>12}")
for B in (0.1, 0.5, 1.0, 3.0, 20.0, 150.0):
    po = posterior_odds(1.0, B)  # equal priors -> prior odds = 1
    print(f"{B:>8.1f} {1.0:>12.2f} {po:>12.2f} {odds_to_prob(po):>12.4f}")

    B_ij   prior odds   post. odds   P(M_i | y)
     0.1         1.00         0.10       0.0909
     0.5         1.00         0.50       0.3333
     1.0         1.00         1.00       0.5000
     3.0         1.00         3.00       0.7500
    20.0         1.00        20.00       0.9524
   150.0         1.00       150.00       0.9934


Note what happened at `B = 1.0`, and what happens for `B < 1`.

Now change the prior odds away from 1 and re-run. **Does $B_{ij}$ itself change?** That is
the observation question 2 above is fishing for.

In [2]:
# Same data (same B), three analysts with different prior convictions.
B = 7.0
for prior_odds, who in ((1.0, "agnostic"), (0.1, "sceptic"), (10.0, "enthusiast")):
    po = posterior_odds(prior_odds, B)
    print(f"{who:>12}: prior odds {prior_odds:5.2f} -> posterior odds {po:6.2f} "
          f"(P(M_i|y) = {odds_to_prob(po):.3f})")

    agnostic: prior odds  1.00 -> posterior odds   7.00 (P(M_i|y) = 0.875)
     sceptic: prior odds  0.10 -> posterior odds   0.70 (P(M_i|y) = 0.412)
  enthusiast: prior odds 10.00 -> posterior odds  70.00 (P(M_i|y) = 0.986)


---
## 2. Three models: how many Bayes factors do you actually need?

Invent three marginal likelihoods and look at the structure of the pairwise factors.

In [3]:
# Entirely made-up marginal likelihoods, purely to expose the algebraic structure.
ml = {"M1": 1.2e-6, "M2": 8.4e-6, "M3": 9.1e-6}

B12 = ml["M1"] / ml["M2"]
B23 = ml["M2"] / ml["M3"]
B13 = ml["M1"] / ml["M3"]

print(f"B12       = {B12:.6f}")
print(f"B23       = {B23:.6f}")
print(f"B13       = {B13:.6f}")
print(f"B12 * B23 = {B12 * B23:.6f}   <- compare with B13")
print()
print(f"B21 = 1/B12 = {1 / B12:.6f}")

B12       = 0.142857
B23       = 0.923077
B13       = 0.131868
B12 * B23 = 0.131868   <- compare with B13

B21 = 1/B12 = 7.000000


Two structural facts fall out of that cell. Write them down in your own words — they are
most of the answer to "how do these compare *three* models", because they tell you how many
numbers you have to report and how the rest are determined.

---
## 3. Calibration: is $B = 7$ a lot?

A Bayes factor is unbounded above and has no natural units, so "large" has to be read off
an agreed scale. Your notes reproduce one (look for the Kass & Raftery table — it is the
one with bands like *not worth more than a bare mention* / *positive* / *strong* / *very
strong*). Find it, then use the cell below to see what those bands do to a posterior
probability when the priors are equal.

In [4]:
# Fill in the band edges from the table in your notes and see what they translate to.
band_edges = [1, 3, 20, 150]  # <- check these against your notes

for B in band_edges:
    print(f"B = {B:6.1f}  ->  P(M_i | y) = {odds_to_prob(B):.4f}   (equal priors, 2 models)")

print("\nWhy might a scale like this be described as subjective / conventional?")

B =    1.0  ->  P(M_i | y) = 0.5000   (equal priors, 2 models)
B =    3.0  ->  P(M_i | y) = 0.7500   (equal priors, 2 models)
B =   20.0  ->  P(M_i | y) = 0.9524   (equal priors, 2 models)
B =  150.0  ->  P(M_i | y) = 0.9934   (equal priors, 2 models)

Why might a scale like this be described as subjective / conventional?


---
## 4. Bayes factors vs. posterior model probabilities

Your part (3) already derives $\mathbb{P}(M_i \mid y) = 1 / \sum_k B_{ki}(y)$ under equal
priors. Check it numerically, and think about what it says about the *relationship* between
the two — this is the bridge between deliverable #2 and deliverable #3, and in your current
draft it is buried in a subordinate clause.

In [5]:
import numpy as np

names = list(ml)
vals = np.array([ml[n] for n in names])

direct = vals / vals.sum()  # normalised marginal likelihoods

print(f"{'model':>6} {'via normalising':>17} {'via Bayes factors':>19}")
for i, name in enumerate(names):
    B_ki = vals / vals[i]           # B_ki = p(y|M_k) / p(y|M_i), for k = 1,2,3
    via_bf = 1.0 / B_ki.sum()
    print(f"{name:>6} {direct[i]:>17.6f} {via_bf:>19.6f}")

print(f"\nsum = {direct.sum():.6f}  <- why must this hold, and what did it assume?")

 model   via normalising   via Bayes factors
    M1          0.064171            0.064171
    M2          0.449198            0.449198
    M3          0.486631            0.486631

sum = 1.000000  <- why must this hold, and what did it assume?


That last question matters for your part (3) as written: you assert $\mathcal{M} = \{M_1, M_2, M_3\}$
is a *partition* of the model space. Make sure you can say why that assumption is needed and
what would break without it — especially now that the eq. (1) definition is stated for a
generic $M$ and no longer carries the range.

---
## Your turn

Write the explanation for deliverable #2 here, in your own words, then paste it into the
report. Target: roughly one short paragraph — it is worth about a mark, and part (a) as a
whole should stay near half a page including the displays.

*(your text here)*

---

### Trims to make at the same time

- Cut the zero–one utility / posterior mode sentence — not asked in (a); it earns its keep
  in part (c)(iii) instead, where you have to identify the preferred model.
- Promote "depend on the data only through the Bayes factors" from a trailing clause to a
  point in its own right — it is the link between #2 and #3.

### Sources to check

- Definition 7.3.2 — the Bayes factor (already cited in your draft).
- The Kass & Raftery calibration table in the same chapter.
- Your notes' treatment of posterior model probabilities for the exhaustive-partition case.